# Animation: Polynomial Degree and Overfitting

We fit polynomial curves of increasing degree to noisy data.

The **true underlying function** is a simple sine wave.  
The **data** is that sine wave plus random noise.

Watch what happens as degree increases:
- **Degree 1:** Underfit — straight line misses the curve
- **Degree 4–6:** Good fit — captures the wave without chasing noise
- **Degree 15:** Overfit — perfectly fits every noisy point, useless on new data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

np.random.seed(3)

# ── Generate noisy sine data ────────────────────────────────────
N = 30
X_data = np.sort(np.random.uniform(0, 2*np.pi, N))
y_data = np.sin(X_data) + np.random.randn(N) * 0.3

X_plot  = np.linspace(0, 2*np.pi, 300)
y_true  = np.sin(X_plot)

# ── Fit polynomials of each degree and record predictions ──────
DEGREES = list(range(1, 16))
train_errors, cv_errors, predictions = [], [], []

for deg in DEGREES:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('ridge', Ridge(alpha=1e-6)),   # tiny regularization for numerical stability
    ])
    pipe.fit(X_data.reshape(-1, 1), y_data)
    y_pred_train = pipe.predict(X_data.reshape(-1, 1))
    y_pred_plot  = pipe.predict(X_plot.reshape(-1, 1))

    train_mse = np.mean((y_pred_train - y_data)**2)
    cv_scores = cross_val_score(pipe, X_data.reshape(-1,1), y_data,
                                cv=5, scoring='neg_mean_squared_error')
    train_errors.append(train_mse)
    cv_errors.append(-cv_scores.mean())
    predictions.append(y_pred_plot)

print('Degrees:', DEGREES)
print('Train MSE (deg 1, 5, 15):', [round(train_errors[i], 4) for i in [0, 4, 14]])
print('CV MSE    (deg 1, 5, 15):', [round(cv_errors[i],    4) for i in [0, 4, 14]])

In [ ]:
# ── Animation: top panel = fitted curve, bottom panel = error ──
fig, (ax_fit, ax_err) = plt.subplots(2, 1, figsize=(8, 9),
                                      gridspec_kw={'height_ratios': [2.2, 1]})
fig.subplots_adjust(hspace=0.35)

# ─ Top: curve fit ───────────────────────────────────────────────
ax_fit.scatter(X_data, y_data, color='#2c3e50', s=50, zorder=4,
               label='Noisy data', alpha=0.8)
ax_fit.plot(X_plot, y_true, '--', color='#95a5a6', lw=2, label='True function (sin x)', zorder=2)

fit_line, = ax_fit.plot([], [], color='#e74c3c', lw=2.5, zorder=3, label='Polynomial fit')
ax_fit.set_xlim(0, 2*np.pi)
ax_fit.set_ylim(-2.5, 2.5)
ax_fit.set_xlabel('x', fontsize=12)
ax_fit.set_ylabel('y', fontsize=12)
ax_fit.legend(loc='upper right', fontsize=10)
title_fit = ax_fit.set_title('', fontsize=13)

# Region label
region_text = ax_fit.text(0.02, 0.95, '', transform=ax_fit.transAxes,
                           fontsize=12, va='top', fontweight='bold',
                           bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.85))

# ─ Bottom: error curve ──────────────────────────────────────────
ax_err.plot(DEGREES, train_errors, 'o-', color='#3498db', lw=2, label='Train MSE')
ax_err.plot(DEGREES, cv_errors,    'o-', color='#e74c3c', lw=2, label='CV MSE')
ax_err.set_xlim(0.5, 15.5)
ax_err.set_ylim(0, max(cv_errors) * 1.15)
ax_err.set_xlabel('Polynomial Degree', fontsize=11)
ax_err.set_ylabel('Mean Squared Error', fontsize=11)
ax_err.set_title('Train vs. Cross-Validation Error', fontsize=12)
ax_err.legend(fontsize=10)
ax_err.set_xticks(DEGREES)
ax_err.grid(True, alpha=0.3)

# Moving marker on error plot
marker_train, = ax_err.plot([], [], 'o', color='#3498db', ms=12, zorder=5)
marker_cv,    = ax_err.plot([], [], 'o', color='#e74c3c', ms=12, zorder=5)
vline = ax_err.axvline(x=0, color='#2c3e50', lw=1.5, linestyle='--', alpha=0.6)

def update(frame):
    deg = DEGREES[frame]
    y_pred = predictions[frame]

    fit_line.set_data(X_plot, y_pred)
    title_fit.set_text(f'Polynomial Fit — Degree {deg}')

    # Region label
    if deg <= 2:
        label, color = 'UNDERFITTING\n(high bias)', '#e67e22'
    elif deg <= 7:
        label, color = 'GOOD FIT', '#27ae60'
    else:
        label, color = 'OVERFITTING\n(high variance)', '#e74c3c'
    region_text.set_text(label)
    region_text.get_bbox_patch().set_facecolor(color)
    region_text.set_color('white')

    # Error markers
    marker_train.set_data([deg], [train_errors[frame]])
    marker_cv.set_data([deg], [cv_errors[frame]])
    vline.set_xdata([deg, deg])

    return fit_line, title_fit, region_text, marker_train, marker_cv, vline

anim = FuncAnimation(fig, update, frames=len(DEGREES),
                     interval=700, repeat=True, blit=True)
plt.close()
HTML(anim.to_jshtml())

## What to notice

**Top panel:** The red curve is our model. Watch it go from too-straight → just right → wildly oscillating.

**Bottom panel:** This is the key insight:
- **Train MSE** always decreases as degree increases — the model fits the training data better and better
- **CV MSE** decreases at first, then *increases* — the model starts memorizing noise and fails on new data
- **The sweet spot** is where CV MSE is lowest (around degree 4–6)

This is exactly **why we cross-validate**: training error alone would always tell you to use degree 15.

---

**Bias-Variance Tradeoff summary:**

| Model | Bias | Variance | Train Error | CV Error |
|-------|------|----------|-------------|----------|
| Degree 1 | High | Low | High | High |
| Degree 4–6 | Low | Low | Medium | **Lowest** |
| Degree 15 | Low | High | Very low | High |